<a href="https://colab.research.google.com/github/kartik815/Amazon-ML-Challenge-2026/blob/main/notebooks/07_Decision_Layer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 07 - S1 Decision Layer and macro F0.5

The pairwise model only gives `P(match | pair)`. The challenge is scored at the
**S1 entity level**: for each S1 we must produce zero, one or several matches.

This notebook:

1. Loads the validation predictions produced by notebook 06.
2. Converts pair probabilities into per-S1 match sets.
3. Tunes the decision rule to maximise the challenge metric
   `macro F0.5` (precision weighted, beta = 0.5).
4. Saves the chosen decision configuration for test inference (notebook 08).

Metric per S1 entity:

```
F0.5 = 1.25 * P * R / (0.25 * P + R)
```

Special cases:

* true empty and predicted empty -> 1.0 (helps unmatched entities)
* true empty and predicted non-empty -> 0.0
* true non-empty and predicted empty -> 0.0

The score is the mean of the per-entity F0.5.


In [ ]:
import os
import json
import gc
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")


In [ ]:
DRIVE_ROOT = "/content/drive/MyDrive/Amazon ML Challenge 2026"
MODEL_ROOT = os.path.join(DRIVE_ROOT, "03_Experiments", "Model")
CLEANED_DATA_ROOT = os.path.join(DRIVE_ROOT, "03_Experiments", "Cleaned_Data")
TRAIN_ROOT = os.path.join(
    DRIVE_ROOT, "01_Dataset", "6ab10eb3b23ba_student_resource",
    "student_resource", "dataset", "train"
)

VAL_PRED_PATH = os.path.join(MODEL_ROOT, "val_predictions.tsv")
GT_PATH = os.path.join(TRAIN_ROOT, "train_ground_truth.tsv")
S1_PATH = os.path.join(CLEANED_DATA_ROOT, "train_s1_cleaned.tsv")
DECISION_PATH = os.path.join(MODEL_ROOT, "decision_config.json")

print("val preds:", os.path.exists(VAL_PRED_PATH))


## Metric and decision rule

In [ ]:
def entity_f05(true_ids, pred_ids):
    tp = len(true_ids & pred_ids)
    fp = len(pred_ids - true_ids)
    fn = len(true_ids - pred_ids)

    if not true_ids and not pred_ids:
        return 1.0
    if not true_ids or not pred_ids:
        return 0.0

    precision = tp / (tp + fp) if (tp + fp) else 0.0
    recall = tp / (tp + fn) if (tp + fn) else 0.0

    if precision + recall == 0:
        return 0.0

    beta2 = 0.25  # beta = 0.5
    return (1 + beta2) * precision * recall / (beta2 * precision + recall)


def decide_entity(group, threshold, relative):
    # group: rows for one S1 with columns match_probability, candidate_entity_id
    probs = group["match_probability"].to_numpy()
    if len(probs) == 0:
        return set()
    cut = max(threshold, relative * float(probs.max()))
    return set(group.loc[probs >= cut, "candidate_entity_id"])


def macro_f05(pred_df, threshold, relative=0.0):
    scores = []
    for _, group in pred_df.groupby("s1_entity_id", sort=False):
        true_ids = set(group.loc[group["label"] == 1, "candidate_entity_id"])
        pred_ids = decide_entity(group, threshold, relative)
        scores.append(entity_f05(true_ids, pred_ids))
    return float(np.mean(scores)) if scores else 0.0


## Threshold sweep (candidate-covered entities)

In [ ]:
val_pred = pd.read_csv(
    VAL_PRED_PATH,
    sep="\t",
    dtype={
        "s1_entity_id": "string",
        "candidate_entity_id": "string",
        "source": "string",
    },
)

print("validation pairs:", len(val_pred))
print("validation S1 entities:", val_pred["s1_entity_id"].nunique())
print(val_pred["label"].value_counts())

results = []
for relative in [0.0, 0.90, 0.95, 0.98]:
    for threshold in [0.5, 0.6, 0.7, 0.8, 0.85, 0.9, 0.93, 0.95, 0.97, 0.98, 0.99, 0.995]:
        results.append({
            "threshold": threshold,
            "relative": relative,
            "macro_f05": macro_f05(val_pred, threshold, relative),
        })

results = pd.DataFrame(results).sort_values("macro_f05", ascending=False)
print(results.head(20).to_string(index=False))


## Full macro F0.5 over all S1 entities (includes empty predictions)

In [ ]:
# The sweep above only counts S1 entities that appear in the candidate file.
# The true challenge metric also includes S1 entities with NO candidate at all.
# Those entities predict the empty set, so they score 1.0 if they truly have no
# match, and 0.0 if blocking dropped all of their true matches.

import pandas as pd

s1_ids = pd.read_csv(S1_PATH, sep="\t", usecols=["entity_id"], dtype="string")["entity_id"]

gt = pd.read_csv(GT_PATH, sep="\t", dtype="string")
gt_lookup = {}
for row in gt.itertuples(index=False):
    m = row.matched_entity_ids
    gt_lookup[row.source1_entity_id] = (
        set() if (pd.isna(m) or m == "") else {x.strip() for x in str(m).split(",") if x.strip()}
    )
del gt
gc.collect()

best = results.iloc[0]
best_thr = float(best["threshold"])
best_rel = float(best["relative"])

by_s1 = {k: v for k, v in val_pred.groupby("s1_entity_id", sort=False)}

scores = []
empty_correct = 0
empty_wrong = 0

for s1_id in s1_ids:
    group = by_s1.get(s1_id)
    true_ids = gt_lookup.get(s1_id, set())

    if group is None:
        pred_ids = set()
    else:
        pred_ids = decide_entity(group, best_thr, best_rel)

    if not true_ids and not pred_ids:
        empty_correct += 1
    elif not true_ids and pred_ids:
        empty_wrong += 1

    scores.append(entity_f05(true_ids, pred_ids))

print("threshold        :", best_thr)
print("relative         :", best_rel)
print("S1 entities total:", len(s1_ids))
print("empty correct    :", empty_correct)
print("empty wrong      :", empty_wrong)
print("TRUE macro F0.5  :", round(float(np.mean(scores)), 6))


## Save decision configuration

In [ ]:
decision_config = {
    "threshold": best_thr,
    "relative": best_rel,
    "note": "cut = max(threshold, relative * max_probability_within_s1)",
}

with open(DECISION_PATH, "w", encoding="utf-8") as f:
    json.dump(decision_config, f, indent=2)

print("saved decision config:", DECISION_PATH)
print(decision_config)
